# SharkByte integrated verification on Google Colab
Use a CPU runtime. The first execution step runs the full 40-qubit, 56-gate sparse benchmark and compares every one of its 16,777,216 populated amplitudes with Cirq. The circuit factors into eight disconnected components. Subsequent cells run the integrated repository checks.
Run the setup cells and the full comparison first. The last cell exports receipts and logs as a ZIP.


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import tempfile

revision = "ec876e2c51844068e01d8fac2ee86b83983514a9"
run_dir = Path(tempfile.mkdtemp(prefix="sharkbyte-colab-"))
repo = run_dir / "SharkByte"
results = run_dir / "receipts"
results.mkdir()

def run(*args):
    env = dict(os.environ, MPLBACKEND="Agg")
    result = subprocess.run([str(arg) for arg in args], cwd=repo, env=env,
                            capture_output=True, text=True)
    print(result.stdout, end="")
    print(result.stderr, end="", file=sys.stderr)
    result.check_returncode()

subprocess.run(["git", "clone", "--quiet", "https://github.com/reggaesharkk/SharkByte.git", str(repo)], check=True)
run("git", "checkout", "--detach", revision)
print(platform.platform(), platform.machine(), sys.version.split()[0])
print("Results:", results)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
if actual != revision:
    raise RuntimeError(f"Expected {revision}, checked out {actual}")


In [ ]:
def run_setup(*args):
    command = [str(arg) for arg in args]
    completed = subprocess.run(command, cwd=repo, capture_output=True, text=True)
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    completed.check_returncode()

def install_env(name, *requirements):
    # A fresh path allows this cell to be retried after partial setup.
    location = Path(tempfile.mkdtemp(prefix=f"{name}-", dir=run_dir))
    run_setup(sys.executable, "-m", "venv", "--without-pip", location)
    python = location / "bin" / "python"
    pip = (sys.executable, "-m", "pip", "--python", python)
    run_setup(*pip, "install", *requirements)
    run_setup(*pip, "check")
    return python


reference_python = install_env("references", "-r", "requirements-high-support.txt")


In [ ]:

results = Path(tempfile.mkdtemp(prefix="integrated-results-", dir=run_dir))
checks = {}

def check(name, *args, cwd=None, timeout=600, requires=()):
    blocked = [dep for dep in requires if checks.get(dep, {}).get("status") != "PASS"]
    if blocked:
        checks[name] = {"status": "NOT RUN", "blocked_by": blocked}
        return
    cmd = [str(arg) for arg in args]
    try:
        proc = subprocess.run(cmd, cwd=cwd or repo, env=dict(os.environ, MPLBACKEND="Agg"),
                              capture_output=True, text=True, errors="replace", timeout=timeout)
        code = proc.returncode
        log = proc.stdout + proc.stderr
        status = "PASS" if code == 0 else "FAIL"
    except subprocess.TimeoutExpired as exc:
        code = None
        log = str(exc) + "\n" + repr(exc.stdout) + "\n" + repr(exc.stderr)
        status = "TIMEOUT"
    (results / f"{name}.log").write_text(log, encoding="utf-8")
    checks[name] = {"status": status, "exit_code": code, "command": cmd}
    print(name, status, flush=True)

py = reference_python


In [ ]:
# Live 40-qubit, 16,777,216-amplitude comparison.
def available_memory_bytes():
    meminfo = Path("/proc/meminfo").read_text(encoding="utf-8")
    available = int(next(line.split()[1] for line in meminfo.splitlines()
                         if line.startswith("MemAvailable:"))) * 1024
    for limit_path, usage_path in (
        ("/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory.current"),
        ("/sys/fs/cgroup/memory/memory.limit_in_bytes",
         "/sys/fs/cgroup/memory/memory.usage_in_bytes"),
    ):
        try:
            limit = Path(limit_path).read_text(encoding="utf-8").strip()
            if limit.isdecimal():
                usage = int(Path(usage_path).read_text(encoding="utf-8"))
                available = min(available, max(0, int(limit) - usage))
        except FileNotFoundError:
            continue
    return available

available = available_memory_bytes()
minimum = int(4.25 * 1024**3)
print(f"Available memory: {available / 1024**3:.2f} GiB")
if available < minimum:
    checks["full-40q-cirq"] = {"status": "REFUSED", "available_bytes": available,
                               "minimum_bytes": minimum}
    print("Full comparison refused. Use a Colab CPU runtime with more available RAM.")
else:
    receipt = results / "cirq-40q-16777216.json"
    check("full-40q-cirq", py, "-B", "tools/run_cirq_scaling_check.py",
          "--output", receipt, timeout=600)
    if checks["full-40q-cirq"]["status"] == "PASS":
        result = json.loads(receipt.read_bytes())
        expected = ("PASS", revision, 40, 56, 16777216, 16777216, 16777216)
        keys = ("status", "revision", "qubits", "gates", "compared_amplitudes",
                "reference_active_states", "sharkbyte_peak_states")
        if tuple(result.get(key) for key in keys) != expected:
            checks["full-40q-cirq"]["status"] = "FAIL"
            print("Receipt did not match the requested circuit or revision.")
        else:
            print("Verified all 16,777,216 populated amplitudes.")
            print("Maximum complex amplitude error:", result["max_complex_amplitude_error"])


In [ ]:
check("checkout-tests", py, "-B", "-m", "unittest", "discover", "-s", "tests", "-v")
check("high-support-tests", py, "-B", "-m", "unittest", "validation.high_support.test_high_support", "-v")
check("studio-browser-receipt", py, "-B", "tools/verify_studio_scaling_receipt.py",
      "evidence/studio_16m/windows_arm64/sharkbyte-85c74d14dcb54040a295e3c2fe422e53.json")
for mode, flag in (("normal", ()), ("O", ("-O",)), ("OO", ("-OO",))):
    check("rotation-" + mode, py, *flag, "-B", "tools/rotation_gate_evidence.py")
check("connected-ghz-analytic", py, "-B", "tools/run_connected_sparse_analytic_validation.py",
      "--output", results / "connected-ghz.json")


In [ ]:

for backend in ("qiskit", "cirq"):
    check(f"connected-{backend}", py, "-B", "tools/run_connected_quantum_validation.py",
          "--backend", backend, "--output", results / f"connected-{backend}.json",
          "--support-cap", 65536, "--memory-budget-mib", 64, "--timeout-seconds", 60)
check("connected-comparison", py, "-B", "tools/compare_connected_quantum_receipts.py",
      results / "connected-qiskit.json", results / "connected-cirq.json",
      "--output", results / "connected-comparison.json",
      requires=("connected-qiskit", "connected-cirq"))


In [ ]:

for case, width in (("random_connected", 17), ("random_connected", 18),
                    ("random_connected", 20), ("connected_separable", 20),
                    ("numerical_adversary", 17), ("roundtrip", 17)):
    label = f"{case}-{width}"
    check(label, py, "-B", "tools/high_support/run_suite.py",
          "--case", case, "--width", width, "--output-dir", results / label,
          "--repeat", 1, "--delete-vectors-after-comparison")


In [ ]:

inventory = {p.relative_to(results).as_posix(): {"bytes": p.stat().st_size,
             "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
             for p in sorted(results.rglob("*")) if p.is_file()}
versions = subprocess.check_output([str(py), "-c",
    "import sys,qiskit,cirq,numpy; print(sys.version.split()[0], qiskit.__version__, cirq.__version__, numpy.__version__)"],
    cwd=repo, env=dict(os.environ, MPLBACKEND="Agg"), text=True).split()
summary = {"revision": revision, "host": platform.platform(), "architecture": platform.machine(),
           "versions": dict(zip(("python", "qiskit", "cirq", "numpy"), versions)),
           "checks": checks, "files": inventory}
(results / "colab-summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print(json.dumps({k: v["status"] for k,v in checks.items()}, indent=2))
archive = shutil.make_archive(str(results), "zip", results)
from google.colab import files
files.download(archive)
